#### <b> RECURRENT NEURAL NETWORK (CNN)</b>

A Recurrent Neural Network (RNN) is a neural network designed to process sequential data, where the output at time 
𝑡 depends not only on the current input but also on previous inputs through a hidden state.

3 issues using ANN for sequence problems:
- Variable size of the ip/op neurons
- Too much computation
- No parameter sharing

A sequence problem deals with data that comes step-by-step in a specific order over time or position.

Core idea:

RNN introduces **memory**
 

Instead of computing:

$$
y = f(Wx + b)
$$

RNN computes:

$$
h_t = f(W_x x_t + W_h h_{t-1} + b)
$$

Where:

- \(x_t\) → input at time \(t\)  
- \(h_t\) → hidden state (memory)  
- \(h_{t-1}\) → previous hidden state  
- \(W_x\) → input weights  
- \(W_h\) → recurrent weights  

So the network **remembers previous information**.

TYPES OF RNN:

> 

> ONE TO MANY:

The model receives one input , then generates a sequence of outputs over time

The hidden state acts as a memory that keeps generating outputs step- by-step

<img src='onetomany.png' width=400/>

applications:
- Image captioning
- Music generation
- Text generation
- Story generation

> MANY TO ONE:

Sequence Input -> Single output

x1 → h1

        ↓

x2 → h2

        ↓

x3 → h3

        ↓

x4 → h4 → y

<img src='manytoone.png' width=400/>

applications:
- Sentiment analysis
- Spam detection
- Document classification
- Emotion detection



> MANY TO MANY

Multiple Inputs -> Multiple Outputs

**TYPE 1: Same-length Many to Many**

The input and output sequence have the same length

**TYPE 2 : Encoder-Decoder(Different length)**

This is used in machine translation

### Key Insight

RNN works because it **shares the same weights across time steps**.

$$
h_t = f(W_x x_t + W_h h_{t-1})
$$

So the network learns **patterns in sequences**.

Applications of RNN:

NLP:
- Text prediction
- Machine Translation
- Chatbot
- Sentiment analysis

Time Series:
- Stock price prediction
- Weather forcasting
- Energy demand prediction

#### LONG SHORT TERM MEMORY (LSTM):

LSTM networks excel at processing sequences such as time series, text, or speech by maintaining a cell state that preserves information over extended periods. They mitigate the vanishing gradient problem through a gating mechanism, allowing the model to selectively remember or forget data.

**Forget Gate:** Decides what information to discard from the cell state using a sigmoid function on the previous hidden state and current input, outputting values between 0 (forget) and 1 (keep).

**Input Gate:** Determines what new information to store, combining a sigmoid layer to select relevant updates and a tanh layer to create candidate values.

**Output Gate:** Controls the output by filtering the cell state through a sigmoid and tanh, producing the final hidden state for the next time step.

#### Simple Application of RNN in NLP, time series

In [2]:
import torch                 #Core PyTorch library for tensors and deep learning
import torch.nn as nn        #Contain neural network modules like RNN, Linear Layers
import torch.optim as optim  #Provides optimization algorithms like Adam,SGD


In [3]:
text = "i love deep learning and i love pytorch"
# A small sentence used as training data for sequence prediction

words = text.split()
# Splits sentence into list of words
# ['i','love','deep','learning','and','i','love','pytorch']

In [4]:
vocab = list(set(words))
# Creates unique list of words (removes duplicates)

word_to_ix = {word:i for i,word in enumerate(vocab)}
# Creates dictionary mapping each word to a unique integer index
# Example: {'i':0,'love':1,'deep':2,...}

In [5]:
vocab_size = len(vocab)
# Total number of unique words in dataset

In [6]:
sequence_length = 2
# Number of words used as input to predict next word

inputs = []
targets = []
# Lists to store training data

In [7]:
for i in range(len(words) - sequence_length):

    seq = words[i:i+sequence_length]
    # Selects input sequence of length 2

    target = words[i+sequence_length]
    # The next word that the model must predict

    inputs.append([word_to_ix[w] for w in seq])
    # Converts words into their index numbers

    targets.append(word_to_ix[target])
    # Stores index of target word

In [8]:
inputs = torch.tensor(inputs)
# Converts input sequences into PyTorch tensor

targets = torch.tensor(targets)
# Converts target outputs into tensor

In [17]:
class TextRNN(nn.Module):
# Creates a neural network class called TextRNN
# nn.Module is the base class for all PyTorch models
        def __init__(self, vocab_size, embedding_dim, hidden_dim):

                super(TextRNN, self).__init__()
                # Initializes the parent class nn.Module

                self.embedding = nn.Embedding(vocab_size, embedding_dim)
                # Converts word indices into dense vectors (word embeddings)

                self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
                # RNN layer
                # embedding_dim = size of word vector
                # hidden_dim = number of hidden neurons

                self.fc = nn.Linear(hidden_dim, vocab_size)
                # Fully connected layer to predict next word
        def forward(self, x):

                x = self.embedding(x)
        # Converts word indices into embedding vectors

                out, hidden = self.rnn(x)
        # Passes sequence through RNN
        # out = output for each time step
        # hidden = final hidden state

                out = out[:, -1, :]
        # Selects output from the last time step

                out = self.fc(out)
        # Converts hidden representation to vocabulary prediction

                return out

In [18]:
embedding_dim = 10
# Size of word embedding vector

hidden_dim = 20
# Number of hidden neurons in RNN

model = TextRNN(vocab_size, embedding_dim, hidden_dim)
# Creates the RNN model

In [19]:
criterion = nn.CrossEntropyLoss()
# Loss function for multi-class classification

optimizer = optim.Adam(model.parameters(), lr=0.01)
# Adam optimizer updates model weights during training

In [20]:
for epoch in range(200):
# Train model for 200 iterations
    outputs = model(inputs)
    # Sends input sequences into the RNN
    loss = criterion(outputs, targets)
    # Compares predicted words with actual target words
    optimizer.zero_grad()
    # Clears previous gradients

    loss.backward()
    # Computes gradients using backpropagation

    optimizer.step()
    # Updates model weights
    if epoch % 20 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())
test = torch.tensor([[word_to_ix['i'], word_to_ix['love']]])
# Input sequence: "i love"
prediction = model(test)
# Model predicts the next word
predicted_word = vocab[torch.argmax(prediction).item()]
# Finds word with highest probability
print("Predicted next word:", predicted_word)

Epoch: 0 Loss: 1.8691538572311401
Epoch: 20 Loss: 0.3192502558231354
Epoch: 40 Loss: 0.24334675073623657
Epoch: 60 Loss: 0.23722267150878906
Epoch: 80 Loss: 0.23545710742473602
Epoch: 100 Loss: 0.23450367152690887
Epoch: 120 Loss: 0.23385603725910187
Epoch: 140 Loss: 0.23338115215301514
Epoch: 160 Loss: 0.23302043974399567
Epoch: 180 Loss: 0.23273968696594238
Predicted next word: deep
